# LankaWeather AI: Ensemble Analysis

This notebook inspects the processed dataset, documents chronological preparation, runs the reusable Stage 3 training/evaluation pipeline, and reviews measured outputs. Run from the repository root after installing `requirements.txt`.

Models are compared on the final chronological 20% only; no shuffling is used.

## Dataset overview and target discovery

The target is discovered from the unique binary 0/1 column in the actual CSV. Date/time, identifiers, the target, and future-looking fields are excluded from predictors.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from train_models import load_dataset, run_pipeline

DATA_PATH = ROOT / 'data' / 'processed' / 'weather_features.csv'
raw = pd.read_csv(DATA_PATH)
X, y, target_name, dates = load_dataset(DATA_PATH)
print(f'Rows: {len(raw):,}; columns: {raw.columns.tolist()}')
print(f'Discovered target: {target_name}; predictors: {list(X.columns)}')
print(f'Date range: {dates.min().date()} through {dates.max().date()}')
display(raw.describe(include='all').T)
display(y.value_counts().rename('count').to_frame())

## Exploratory data analysis and preprocessing

The source has daily weather features and the next-day binary rain label. The training script sorts by date, applies a strict chronological 80/20 split, one-hot encodes categorical predictors using training categories, and does not impute missing data. Rainfall class proportions and train/test periods are printed during execution.

In [ ]:
ax = y.value_counts(normalize=True).sort_index().plot(kind='bar', color=['steelblue', 'teal'])
ax.set(title='Target class proportions', xlabel='Next-day rain class', ylabel='Proportion', xticklabels=['0: no rain', '1: rain'])
plt.tight_layout()
plt.show()

## Ensemble approaches, evaluation, and charts

The reusable pipeline trains a most-frequent dummy baseline, balanced-subsample Random Forest (bagging), training-weighted Gradient Boosting, expanding-window out-of-fold stacking (RF + GB with logistic regression meta learner), and soft voting hybrid. Every model uses the identical untouched test set. It writes metrics, classification reports, confusion matrices, ROC/PR curves, and tree feature importances.

In [ ]:
# Executes the complete reproducible training and evaluation run.
results = run_pipeline()
display(results[['model', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'balanced_accuracy', 'training_time_seconds', 'prediction_time_seconds']])

## Findings and conclusion

The following reads the actual results written by the training run. Highest scores are computed from the measured table; no values are assumed. See `results/model_analysis.md` for interpretation, class imbalance, comparison of stacking/hybridization, and limitations.

In [ ]:
from IPython.display import Markdown, display
analysis_path = ROOT / 'results' / 'model_analysis.md'
metrics = pd.read_csv(ROOT / 'results' / 'model_comparison.csv')
display(metrics.set_index('model')[['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'balanced_accuracy']])
display(Markdown(analysis_path.read_text(encoding='utf-8')))
for image in sorted((ROOT / 'results' / 'figures').glob('*.png')):
    display(Markdown(f'### {image.stem.title()}'))
    display(plt.imread(image))